# Robotic Warehouse Simulation Scenario Analysis

This notebook analyzes a discrete-event simulation of a robotic warehouse fleet. The objective is to evaluate throughput, latency, SLA attainment, fleet utilization, and bottlenecks across operating scenarios.

**Executive question:** before changing a real robotic warehouse, can we estimate whether demand growth, fleet size, or station capacity will become the next bottleneck?

## Modeling assumptions

- Orders arrive stochastically using a Poisson-style arrival process.
- Robots pull orders from a shared queue.
- Pick/drop stations are limited resources.
- Travel, pick, drop-off, repair, and charging times are stochastic.
- SLA attainment is measured as the share of orders completed within 20 minutes.
- The simulation is designed for scenario comparison, not exact production forecasting until calibrated with real data.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = PROJECT_ROOT / 'src'
sys.path.append(str(SRC))

from warehouse_sim.simulation import run_scenarios, SimConfig

REPORTS_DIR = PROJECT_ROOT / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
REPORTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

## Run base scenarios

The default scenarios compare baseline operations, smaller/larger fleets, higher demand, and additional station capacity.

In [ ]:
orders, queues, summary = run_scenarios()
display_cols = [
    'scenario_name', 'robot_count', 'station_count', 'arrival_rate_per_minute',
    'completed_orders', 'throughput_per_hour', 'avg_cycle_time_minutes',
    'p90_cycle_time_minutes', 'avg_queue_wait_minutes', 'robot_utilization_proxy',
    'station_utilization_proxy', 'sla_attainment_rate'
]
summary[display_cols].round(2)

## Fleet-size sweep

This test keeps demand and station count fixed, then increases fleet size. The expected executive insight is diminishing returns: after enough robots are available, the next bottleneck becomes station capacity or process time.

In [ ]:
fleet_configs = [
    SimConfig(scenario_name=f'fleet_{robots}', robot_count=robots, station_count=4, seed=100 + robots)
    for robots in [10, 15, 20, 25, 30, 40]
]
_, _, fleet_summary = run_scenarios(fleet_configs)
fleet_summary[['robot_count', 'throughput_per_hour', 'avg_cycle_time_minutes', 'sla_attainment_rate']].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(fleet_summary['robot_count'], fleet_summary['throughput_per_hour'], marker='o')
ax.set_title('Throughput improves then plateaus as fleet size grows')
ax.set_xlabel('Robot fleet size')
ax.set_ylabel('Completed orders per hour')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'throughput_by_fleet_size.svg')
plt.show()

## Demand stress test

This test increases order arrival rate while keeping robot and station capacity fixed. The key signal is the point where cycle time and P90 latency grow faster than throughput.

In [ ]:
demand_configs = [
    SimConfig(scenario_name='demand_0_70', order_arrival_rate_per_minute=0.70, seed=201),
    SimConfig(scenario_name='demand_0_80_baseline', order_arrival_rate_per_minute=0.80, seed=202),
    SimConfig(scenario_name='demand_1_00', order_arrival_rate_per_minute=1.00, seed=203),
    SimConfig(scenario_name='demand_1_20', order_arrival_rate_per_minute=1.20, seed=204),
    SimConfig(scenario_name='demand_1_40', order_arrival_rate_per_minute=1.40, seed=205),
]
_, _, demand_summary = run_scenarios(demand_configs)
demand_summary[['arrival_rate_per_minute', 'throughput_per_hour', 'avg_cycle_time_minutes', 'p90_cycle_time_minutes', 'sla_attainment_rate']].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(demand_summary['arrival_rate_per_minute'], demand_summary['avg_cycle_time_minutes'], marker='o', label='Average cycle time')
ax.plot(demand_summary['arrival_rate_per_minute'], demand_summary['p90_cycle_time_minutes'], marker='o', label='P90 cycle time')
ax.set_title('Cycle time rises quickly once demand approaches capacity')
ax.set_xlabel('Order arrival rate per minute')
ax.set_ylabel('Cycle time minutes')
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'cycle_time_by_demand.svg')
plt.show()

## Executive recommendation

The simulation shows that simply adding robots is not always the best investment. When demand rises, latency and SLA risk increase quickly. After the fleet is large enough, station capacity and process time become the more important constraints.

**Recommended next decision:** before expanding fleet size, run a combined scenario that tests station capacity, robot count, demand growth, charging rules, and failure rate together. The production version of this model should be calibrated using actual order arrival, travel-time, pick-time, downtime, and charge-cycle data.